In [19]:
import argparse
from pathlib import Path

# Create Class

In [20]:
class create_files:
    def __init__(self, m: int):
        self.m = m
        self.entity_name = f"sum{m}xn"

    def is_power_of_two(self, x: int) -> bool:
        return x >= 2 and (x & (x - 1)) == 0


    def chunks(self, lista, tamanho):
        for i in range(0, len(lista), tamanho):
            yield lista[i:i + tamanho]


    def format_port_list(self, ports, indent="        ", per_line=4):
        lines = []

        for group in self.chunks(ports, per_line):
            lines.append(indent + ", ".join(group))

        return ",\n".join(lines)

    def generate_sum_vhdl(self, m: int) -> str:
        if not self.is_power_of_two(m):
            raise ValueError("O número de vetores M deve ser potência de 2: 2, 4, 8, 16, 32...")

        entity_name = f"sum{m}xn"
        ports = [f"a{i}" for i in range(m)]

        vhdl = []

        vhdl.append(f"""library ieee;
    use ieee.std_logic_1164.all;
    use ieee.numeric_std.all;

    entity {entity_name} is
        generic(
            n : integer := 16
        );
        port (
            {self.format_port_list(ports)} : in unsigned(n-1 downto 0);
            sum : out unsigned(n-1 downto 0)
        );
    end {entity_name};

    architecture rtl of {entity_name} is
    """)

        # ------------------------------------------------------------
        # Nível 1: sinais diagonais
        # Cada entrada ai é deslocada i posições para a esquerda.
        # A largura continua sendo n bits.
        # Portanto, qualquer bit que passar de n-1 é descartado.
        # ------------------------------------------------------------
        diag_signals = [f"flsb_1_{i}" for i in range(m)]

        vhdl.append("    -- nível 1: deslocamento diagonal")
        for group in self.chunks(diag_signals, 4):
            vhdl.append(
                f"    signal {', '.join(group)} : unsigned(n-1 downto 0);"
            )
        vhdl.append("")

        # ------------------------------------------------------------
        # Declaração dos sinais intermediários da árvore de soma
        # ------------------------------------------------------------
        current_signals = diag_signals.copy()
        levels = []
        level = 2

        while len(current_signals) > 2:
            next_count = len(current_signals) // 2
            next_signals = [f"flsb_{level}_{i}" for i in range(next_count)]

            levels.append((level, current_signals, next_signals))

            vhdl.append(f"    -- nível {level}")
            for group in self.chunks(next_signals, 4):
                vhdl.append(
                    f"    signal {', '.join(group)} : unsigned(n-1 downto 0);"
                )
            vhdl.append("")

            current_signals = next_signals
            level += 1

        vhdl.append("begin")
        vhdl.append("")

        # ------------------------------------------------------------
        # Atribuições diagonais
        # ------------------------------------------------------------
        vhdl.append("    --------------------------------------------------------------------------")
        vhdl.append("    -- Nível 1: deslocamento diagonal")
        vhdl.append("    --------------------------------------------------------------------------")

        for i in range(m):
            vhdl.append(
                f"    flsb_1_{i} <= shift_left(a{i}, {i});"
            )

        vhdl.append("")

        # ------------------------------------------------------------
        # Atribuições dos níveis de soma
        # ------------------------------------------------------------
        for level, inputs, outputs in levels:
            vhdl.append("    --------------------------------------------------------------------------")
            vhdl.append(f"    -- Nível {level} ({len(outputs)} somas)")
            vhdl.append("    --------------------------------------------------------------------------")

            for i, out_sig in enumerate(outputs):
                in0 = inputs[2 * i]
                in1 = inputs[2 * i + 1]
                vhdl.append(f"    {out_sig} <= {in0} + {in1};")

            vhdl.append("")

        # ------------------------------------------------------------
        # Resultado final
        # ------------------------------------------------------------
        vhdl.append("    --------------------------------------------------------------------------")
        vhdl.append("    -- Resultado final")
        vhdl.append("    --------------------------------------------------------------------------")

        if len(current_signals) == 1:
            vhdl.append(f"    sum <= {current_signals[0]};")
        else:
            vhdl.append(f"    sum <= {current_signals[0]} + {current_signals[1]};")

        vhdl.append("")
        vhdl.append("end architecture;")

        return "\n".join(vhdl)
    
    def generate_pins(self, m: int) -> str:
        if not self.is_power_of_two(m):
            raise ValueError("O número de vetores M deve ser potência de 2: 2, 4, 8, 16, 32...")

        entity_name = f"sum{m}xn"
        ports = [f"a{i}" for i in range(m)]

        qsf  = []

        qsf.append(
    f"""
set_global_assignment -name FAMILY "Cyclone V"
set_global_assignment -name DEVICE 5CGXFC9E7F35C8
set_global_assignment -name TOP_LEVEL_ENTITY {self.entity_name}
set_global_assignment -name ORIGINAL_QUARTUS_VERSION 25.1STD.0
set_global_assignment -name PROJECT_CREATION_TIME_DATE "21:29:52  MAY 02, 2026"
set_global_assignment -name LAST_QUARTUS_VERSION "25.1std.0 Lite Edition"
set_global_assignment -name PROJECT_OUTPUT_DIRECTORY output_files
set_global_assignment -name MIN_CORE_JUNCTION_TEMP 0
set_global_assignment -name MAX_CORE_JUNCTION_TEMP 85
set_global_assignment -name ERROR_CHECK_FREQUENCY_DIVISOR 256
set_global_assignment -name EDA_SIMULATION_TOOL "Questa Altera FPGA (Verilog)"
set_global_assignment -name EDA_TIME_SCALE "1 ps" -section_id eda_simulation
set_global_assignment -name EDA_OUTPUT_DATA_FORMAT "VERILOG HDL" -section_id eda_simulation
set_global_assignment -name EDA_GENERATE_FUNCTIONAL_NETLIST OFF -section_id eda_board_design_timing
set_global_assignment -name EDA_GENERATE_FUNCTIONAL_NETLIST OFF -section_id eda_board_design_symbol
set_global_assignment -name EDA_GENERATE_FUNCTIONAL_NETLIST OFF -section_id eda_board_design_signal_integrity
set_global_assignment -name EDA_GENERATE_FUNCTIONAL_NETLIST OFF -section_id eda_board_design_boundary_scan
set_global_assignment -name PARTITION_NETLIST_TYPE SOURCE -section_id Top
set_global_assignment -name PARTITION_FITTER_PRESERVATION_LEVEL PLACEMENT_AND_ROUTING -section_id Top
set_global_assignment -name PARTITION_COLOR 16764057 -section_id Top"""
        )

        for i in ports:
            qsf.append(f"set_instance_assignment -name VIRTUAL_PIN ON -to {i}")

        qsf.append(
    f"""
set_instance_assignment -name VIRTUAL_PIN ON -to sum
set_global_assignment -name TCL_SCRIPT_FILE max_ff.tcl
set_global_assignment -name VHDL_FILE vhdl/{self.entity_name}.vhd
set_global_assignment -name POWER_PRESET_COOLING_SOLUTION "23 MM HEAT SINK WITH 200 LFPM AIRFLOW"
set_global_assignment -name POWER_BOARD_THERMAL_MODEL "NONE (CONSERVATIVE)"
set_instance_assignment -name PARTITION_HIERARCHY root_partition -to | -section_id Top"""
        )
        
        return "\n".join(qsf)

    def generate_pins_2(self, m: int) -> str:
        if not self.is_power_of_two(m):
            raise ValueError("O número de vetores M deve ser potência de 2: 2, 4, 8, 16, 32...")

        entity_name = f"sum{m}xn"
        ports = [f"a{i}" for i in range(m)]

        qsf = []

        qsf.append(
f"""
set_global_assignment -name FAMILY "Stratix V"
set_global_assignment -name DEVICE 5SEEBH40C2
set_global_assignment -name TOP_LEVEL_ENTITY {entity_name}
set_global_assignment -name ORIGINAL_QUARTUS_VERSION 25.1STD.0
set_global_assignment -name PROJECT_CREATION_TIME_DATE "21:29:52  MAY 02, 2026"
set_global_assignment -name LAST_QUARTUS_VERSION "25.1std.0 Standard Edition"
set_global_assignment -name PROJECT_OUTPUT_DIRECTORY output_files
set_global_assignment -name MIN_CORE_JUNCTION_TEMP 0
set_global_assignment -name MAX_CORE_JUNCTION_TEMP 85
set_global_assignment -name ERROR_CHECK_FREQUENCY_DIVISOR 256

set_global_assignment -name EDA_SIMULATION_TOOL "Questa Altera FPGA (Verilog)"
set_global_assignment -name EDA_TIME_SCALE "1 ps" -section_id eda_simulation
set_global_assignment -name EDA_OUTPUT_DATA_FORMAT "VERILOG HDL" -section_id eda_simulation

set_global_assignment -name EDA_GENERATE_FUNCTIONAL_NETLIST OFF -section_id eda_board_design_timing
set_global_assignment -name EDA_GENERATE_FUNCTIONAL_NETLIST OFF -section_id eda_board_design_symbol
set_global_assignment -name EDA_GENERATE_FUNCTIONAL_NETLIST OFF -section_id eda_board_design_signal_integrity
set_global_assignment -name EDA_GENERATE_FUNCTIONAL_NETLIST OFF -section_id eda_board_design_boundary_scan

set_global_assignment -name PARTITION_NETLIST_TYPE SOURCE -section_id Top
set_global_assignment -name PARTITION_FITTER_PRESERVATION_LEVEL PLACEMENT_AND_ROUTING -section_id Top
set_global_assignment -name PARTITION_COLOR 16764057 -section_id Top
"""
    )

        for port in ports:
            qsf.append(f"set_instance_assignment -name VIRTUAL_PIN ON -to {port}")

        qsf.append(
f"""
set_instance_assignment -name VIRTUAL_PIN ON -to sum

set_global_assignment -name TCL_SCRIPT_FILE max_ff.tcl
set_global_assignment -name TCL_SCRIPT_FILE script.tcl
set_global_assignment -name VHDL_FILE vhdl/{entity_name}.vhd

set_global_assignment -name POWER_PRESET_COOLING_SOLUTION "23 MM HEAT SINK WITH 200 LFPM AIRFLOW"
set_global_assignment -name POWER_BOARD_THERMAL_MODEL "NONE (CONSERVATIVE)"

set_instance_assignment -name PARTITION_HIERARCHY root_partition -to | -section_id Top
"""
    )

        return "\n".join(qsf)

        
    
    def create_files(self, path: str):
        output_dir = Path(path)
        output_dir.mkdir(parents=True, exist_ok=True)
        (output_dir / "vhdl").mkdir(parents=True, exist_ok=True)

        vhdl_content = self.generate_sum_vhdl(self.m)
        qsf_content = self.generate_pins_2(self.m)

        (output_dir / "vhdl" / f"{self.entity_name}.vhd").write_text(vhdl_content, encoding="utf-8")
        (output_dir / f"generic_propose_tree.qsf").write_text(qsf_content, encoding="utf-8")

In [21]:
# x = create_files(32)
# x.create_files(".")

In [22]:
base_dir = Path(r"C:\Users\Rafa\Desktop\HCR\Generics_Tree\Original_Tree_diagonal")

for m in [16, 32, 64, 128, 256, 512]:
    target = base_dir / f"v{m}"
    create_files(m).create_files(target)